# RQ1 — Can we predict purchase amount from customer demographics and browsing behaviour?

**Research Question:** How effectively can baseline supervised learning models (Linear Regression, Decision Tree, KNN) predict `Purchase_Amount` from customer demographics and behavioural features?

**Task:** Regression to predict `Purchase_Amount` (USD).  
**Dataset:** Global E-Commerce Dataset — https://www.kaggle.com/datasets/akrambelha/global-e-commerce-dataset-1m-records  
**Outputs:** CSV metrics table + PDF figure saved to `./outputs/`

## Methodology
1. Load `global_ecommerce.csv` (1M rows, 9 columns) or use built-in simulation.
2. Encode categoricals; impute any missing values.
3. Train/test split 80/20, `random_state=42`.
4. Train three baseline models: LinearRegression, DecisionTreeRegressor, KNeighborsRegressor.
5. Report **MAE**, **RMSE**, **R²**; save table and figure.

In [1]:
# ── Setup & Data Load ──────────────────────────────────────────────────────────
from __future__ import annotations
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RQ_PREFIX   = 'RQ01'
TARGET      = 'Purchase_Amount'
RANDOM_STATE = 42
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)

plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,'font.size':11,
                     'axes.titlesize':13,'axes.labelsize':11})
sns.set_theme(style='whitegrid', context='notebook')

def regression_metrics(y_true, y_pred):
    return {'MAE':  float(mean_absolute_error(y_true, y_pred)),
            'RMSE': float(mean_squared_error(y_true, y_pred)**0.5),
            'R2':   float(r2_score(y_true, y_pred))}

# ── Load or simulate dataset ──────────────────────────────────────────────────
CSV_PATH = Path('global_ecommerce.csv')
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
else:
    np.random.seed(RANDOM_STATE)
    N = 100_000
    cats    = ['Electronics','Clothing','Books','Home & Garden','Sports','Beauty','Toys','Food']
    pays    = ['Credit Card','Debit Card','PayPal','Bank Transfer','Crypto']
    cm      = {'Electronics':2.5,'Clothing':1.1,'Books':0.6,'Home & Garden':1.4,
               'Sports':1.2,'Beauty':0.9,'Toys':0.8,'Food':0.5}
    age     = np.random.randint(18,70,N)
    cat     = np.random.choice(cats, N)
    ni      = np.random.randint(1,10,N)
    bt      = np.random.exponential(15,N).clip(1,120).astype(int)
    base    = np.random.lognormal(3.5,0.8,N)
    pa      = (base*np.array([cm[c] for c in cat])*ni*(1+age/200)+np.random.normal(0,10,N)).clip(5,5000).round(2)
    df = pd.DataFrame({
        'Customer_Age':age,'Gender':np.random.choice(['Male','Female','Other'],N,p=[0.48,0.48,0.04]),
        'Country':np.random.choice(['USA','UK','Germany','France','India','Brazil','Canada','Australia'],N),
        'Product_Category':cat,'Payment_Method':np.random.choice(pays,N,p=[0.35,0.25,0.20,0.15,0.05]),
        'Device':np.random.choice(['Mobile','Desktop','Tablet'],N,p=[0.55,0.35,0.10]),
        'Num_Items':ni,'Browse_Time_Min':bt,'Purchase_Amount':pa})

FEATURE_COLS = [c for c in df.columns if c != TARGET]
X = df[FEATURE_COLS]; y = df[TARGET]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=RANDOM_STATE)

num_cols = ['Customer_Age','Num_Items','Browse_Time_Min']
cat_cols = ['Gender','Country','Product_Category','Payment_Method','Device']
num_cols = [c for c in num_cols if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]

def make_preprocessor(scale=True):
    num_steps = [('imp', SimpleImputer(strategy='median'))]
    if scale: num_steps.append(('sc', StandardScaler()))
    cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                         ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])
    return ColumnTransformer([('num', Pipeline(num_steps), num_cols),
                               ('cat', cat_pipe, cat_cols)])

print(f'Dataset shape: {df.shape}')
print(f'Target: {TARGET}  |  Features: {FEATURE_COLS}')
print(f'Train: {len(X_train)} rows  |  Test: {len(X_test)} rows')

Dataset shape: (100000, 9)
Target: Purchase_Amount  |  Features: ['Customer_Age', 'Gender', 'Country', 'Product_Category', 'Payment_Method', 'Device', 'Num_Items', 'Browse_Time_Min']
Train: 80000 rows  |  Test: 20000 rows


In [2]:
# ── RQ1: Train Baseline Models & Evaluate ─────────────────────────────────────
baseline_models = [
    ('LinearRegression',    LinearRegression()),
    ('DecisionTree',        DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=12, min_samples_leaf=10)),
    ('KNeighbors',          KNeighborsRegressor(n_neighbors=7, weights='distance')),
]

rows = []
for name, model in baseline_models:
    pipe = Pipeline([('prep', make_preprocessor(scale=True)), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    m = regression_metrics(y_test, pred)
    rows.append({'Model': name, **m})
    print(f'{name:22s}  MAE={m["MAE"]:.2f}  RMSE={m["RMSE"]:.2f}  R2={m["R2"]:.4f}')

tbl = pd.DataFrame(rows)
tbl.to_csv(OUT / f'{RQ_PREFIX}_table_baseline_performance.csv', index=False)
print(f'Saved: {OUT}/{RQ_PREFIX}_table_baseline_performance.csv')

# ── Figure ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(tbl)); w = 0.25
ax.bar(x-w, tbl['MAE'],  width=w, label='MAE  (lower=better)', color='steelblue')
ax.bar(x,   tbl['RMSE'], width=w, label='RMSE (lower=better)', color='coral')
ax.bar(x+w, tbl['R2'],   width=w, label='R²   (higher=better)', color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(tbl['Model'], rotation=15, ha='right')
ax.set_ylabel('Metric value')
ax.set_title('RQ1 — Baseline models: MAE, RMSE, R² (test set)')
ax.legend(); plt.tight_layout()
fig.savefig(OUT / f'{RQ_PREFIX}_fig_baseline_comparison.pdf')
plt.show()
print(f'Saved: {OUT}/{RQ_PREFIX}_fig_baseline_comparison.pdf')

LinearRegression        MAE=222.88  RMSE=388.22  R2=0.1399
DecisionTree            MAE=200.37  RMSE=361.28  R2=0.2552
KNeighbors              MAE=212.77  RMSE=377.91  R2=0.1850
Saved: outputs/RQ01_table_baseline_performance.csv
Saved: outputs/RQ01_fig_baseline_comparison.pdf
